#Installation & Setup

In [1]:
!pip install streamlit pyngrok librosa tensorflow torch torchvision pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 62.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 108.4 MB/s eta 0:00:00


#Uploading image,audio and text saved models

In [2]:
from google.colab import files

uploaded = files.upload()

Saving audio_deepfake_final_88.keras to audio_deepfake_final_88.keras


In [3]:
from google.colab import files

uploaded = files.upload()

Saving deepfake_detector_final.pth to deepfake_detector_final.pth


In [9]:
from google.colab import files

uploaded = files.upload()

Saving model.safetensors to model.safetensors


In [10]:
from google.colab import files

uploaded = files.upload()

Saving tokenizer.json to tokenizer.json


In [11]:
from google.colab import files

uploaded = files.upload()

Saving tokenizer_config.json to tokenizer_config.json


In [12]:
from google.colab import files

uploaded = files.upload()

Saving config.json to config.json


#Checking the contents in my listdir

In [13]:
import os

print(os.listdir())

['.config', 'audio_deepfake_final_88.keras', 'config.json', 'logs.txt', 'deepfake_detector_final.pth', 'app.py', 'temp.wav', 'model.safetensors', 'tokenizer_config.json', 'tokenizer.json', 'sample_data']


#Moving all text saved models into a folder

In [14]:
import os
import shutil

os.makedirs("human_ai_detector_final", exist_ok=True)

shutil.move("config.json", "human_ai_detector_final/config.json")
shutil.move("model.safetensors", "human_ai_detector_final/model.safetensors")
shutil.move("tokenizer.json", "human_ai_detector_final/tokenizer.json")
shutil.move("tokenizer_config.json", "human_ai_detector_final/tokenizer_config.json")

print("✅ Files moved successfully")

✅ Files moved successfully


#Testing audio and image streamlit

In [5]:
%%writefile app.py
import streamlit as st
import torch
import torch.nn as nn
import numpy as np
import cv2
import librosa
import tensorflow as tf
import matplotlib.pyplot as plt
from torchvision import models, transforms
from PIL import Image

# =========================
# DEVICE
# =========================
device = torch.device("cpu")

# =========================
# AUDIO MODEL
# =========================
audio_model = tf.keras.models.load_model("audio_deepfake_final_88.keras")

# =========================
# IMAGE MODEL
# =========================
image_model = models.resnet50(weights=None)

image_model.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(image_model.fc.in_features, 256),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(256, 2)
)

image_model.load_state_dict(torch.load("deepfake_detector_final.pth", map_location=device))
image_model.to(device)
image_model.eval()

# =========================
# IMAGE TRANSFORM
# =========================
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5],
                         [0.5, 0.5, 0.5])
])

idx_to_class = {0: "Real", 1: "Fake"}

# =========================
# AUDIO FEATURE EXTRACTION
# =========================
def extract_audio(file_path):
    audio, sr = librosa.load(file_path, sr=16000)
    audio = librosa.util.fix_length(audio, size=32000)

    mel = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=128)
    mel = librosa.power_to_db(mel)

    mel = (mel - np.min(mel)) / (np.max(mel) - np.min(mel) + 1e-6)

    mel = tf.image.resize(mel[..., np.newaxis], (128, 128))

    return np.expand_dims(mel.numpy(), axis=0)

# =========================
# IMAGE PREDICTION
# =========================
def predict_image(image):
    img = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = image_model(img)
        probs = torch.softmax(outputs, dim=1)

    pred = torch.argmax(probs, dim=1).item()
    label = idx_to_class[pred]
    conf = probs[0][pred].item() * 100

    return label, conf, img, pred

# =========================
# GRAD-CAM (IMAGE)
# =========================
def gradcam(img_tensor, class_idx):

    grads, acts = [], []

    def f_hook(m, i, o):
        acts.append(o)

    def b_hook(m, gi, go):
        grads.append(go[0])

    layer = image_model.layer4[-1].conv3

    fh = layer.register_forward_hook(f_hook)
    bh = layer.register_full_backward_hook(b_hook)

    out = image_model(img_tensor)

    image_model.zero_grad()
    out[0][class_idx].backward()

    g = grads[0].detach()
    a = acts[0].detach()

    w = torch.mean(g, dim=[0, 2, 3])

    for i in range(a.shape[1]):
        a[:, i, :, :] *= w[i]

    heatmap = torch.mean(a, dim=1).squeeze()
    heatmap = np.maximum(heatmap.cpu().numpy(), 0)

    if np.max(heatmap) != 0:
        heatmap /= np.max(heatmap)

    fh.remove()
    bh.remove()

    return heatmap

# =========================
# AUDIO SALIENCY (FIXED EXPLANATION METHOD)
# =========================
def audio_saliency(model, input_data):

    input_tensor = tf.convert_to_tensor(input_data)

    with tf.GradientTape() as tape:
        tape.watch(input_tensor)
        pred = model(input_tensor)

    grad = tape.gradient(pred, input_tensor)

    saliency = tf.abs(grad)[0]

    return saliency.numpy().squeeze()

# =========================
# IMAGE REASONING
# =========================
def image_reason(label):

    if label == "Real":
        return """
🟢 WHY REAL IMAGE?

✔ Natural facial texture consistency
✔ Smooth lighting and shadows
✔ No blending artifacts
✔ Realistic skin structure detected
"""
    else:
        return """
🔴 WHY FAKE IMAGE?

❌ Face blending artifacts detected
❌ Irregular texture patterns
❌ Inconsistent lighting
❌ GAN-style manipulation signals
"""

# =========================
# AUDIO REASONING
# =========================
def audio_reason(label):

    if label == "Real":
        return """
🟢 WHY REAL AUDIO?

✔ Stable waveform patterns
✔ Natural speech rhythm
✔ Clean frequency distribution
✔ No synthetic distortions detected
"""
    elif label == "Fake":
        return """
🔴 WHY FAKE AUDIO?

❌ Synthetic frequency artifacts
❌ Irregular spectrogram patterns
❌ AI-generated speech inconsistencies
❌ Distorted energy distribution
"""
    else:
        return "⚠ Uncertain prediction due to mixed signal patterns."

# =========================
# UI
# =========================
st.title("🎯 DeepFake Detection System (Audio + Image)")
st.write("Upload files and get predictions + explanations")

mode = st.selectbox("Select Mode", ["Audio 🎧", "Image 🖼️"])
file = st.file_uploader("Upload File")

# =========================
# AUDIO MODE
# =========================
if mode == "Audio 🎧":

    if file is not None:

        with open("temp.wav", "wb") as f:
            f.write(file.read())

        st.audio("temp.wav")

        features = extract_audio("temp.wav")
        pred = audio_model.predict(features)[0][0]

        st.subheader("🎧 Prediction Score")
        st.write(float(pred))

        if pred < 0.45:
            label = "Real"
            st.success("🟢 REAL AUDIO")
        elif pred > 0.60:
            label = "Fake"
            st.error("🔴 FAKE AUDIO")
        else:
            label = "Uncertain"
            st.warning("⚠ UNCERTAIN")

        # =========================
        # AUDIO EXPLANATION (SALIENCY)
        # =========================
        if st.button("🔍 Explain Audio"):

            saliency = audio_saliency(audio_model, features)

            st.subheader("🎧 Audio Explanation Heatmap")

            fig, ax = plt.subplots()
            im = ax.imshow(saliency, cmap="coolwarm", aspect="auto")

            ax.set_title("Important Audio Regions")
            plt.colorbar(im)

            st.pyplot(fig)

        # TEXT REASON
        st.subheader("🧠 Why this prediction?")
        st.info(audio_reason(label))

# =========================
# IMAGE MODE
# =========================
if mode == "Image 🖼️":

    if file is not None:

        image = Image.open(file).convert("RGB")
        st.image(image, use_container_width=True)

        label, conf, img_tensor, pred_class = predict_image(image)

        st.subheader("🖼️ Prediction")

        if label == "Fake":
            st.error(f"🔴 {label} ({conf:.2f}%)")
        else:
            st.success(f"🟢 {label} ({conf:.2f}%)")

        # =========================
        # GRAD-CAM
        # =========================
        if st.button("🔥 Grad-CAM"):

            heatmap = gradcam(img_tensor, pred_class)

            img = np.array(image.resize((224, 224)))

            heatmap = cv2.resize(heatmap, (224, 224))
            heatmap = np.uint8(255 * heatmap)
            heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)

            overlay = cv2.addWeighted(img, 0.6, heatmap, 0.4, 0)

            st.image(overlay, caption="Grad-CAM Heatmap")

        # TEXT REASON
        st.subheader("🧠 Why this prediction?")
        st.info(image_reason(label))

Writing app.py


#Complete audio+image+text streamlit code

In [15]:
%%writefile app.py

import streamlit as st
import torch
import torch.nn as nn
import numpy as np
import cv2
import librosa
import tensorflow as tf
import matplotlib.pyplot as plt
import shap

from torchvision import models, transforms
from PIL import Image

from transformers import (
    BertTokenizer,
    BertForSequenceClassification
)

# =========================================================
# PAGE CONFIG
# =========================================================
st.set_page_config(
    page_title="Multimodal DeepFake Detector",
    layout="wide"
)

# =========================================================
# DEVICE
# =========================================================
device = torch.device("cpu")

# =========================================================
# TITLE
# =========================================================
st.title("🎯 Multimodal DeepFake Detection System")

st.markdown("""
Detect:
- 🎧 Audio Deepfakes
- 🖼️ Fake Images
- 📝 AI Generated Text
""")

# =========================================================
# AUDIO MODEL
# =========================================================
@st.cache_resource
def load_audio_model():
    return tf.keras.models.load_model(
        "audio_deepfake_final_88.keras"
    )

audio_model = load_audio_model()

# =========================================================
# IMAGE MODEL
# =========================================================
@st.cache_resource
def load_image_model():

    model = models.resnet50(weights=None)

    model.fc = nn.Sequential(
        nn.Dropout(0.5),
        nn.Linear(model.fc.in_features, 256),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(256, 2)
    )

    model.load_state_dict(
        torch.load(
            "deepfake_detector_final.pth",
            map_location=device
        )
    )

    model.to(device)
    model.eval()

    return model

image_model = load_image_model()

# =========================================================
# TEXT MODEL
# =========================================================
TEXT_MODEL_PATH = "human_ai_detector_final"

@st.cache_resource
def load_text_model():

    tokenizer = BertTokenizer.from_pretrained(
        TEXT_MODEL_PATH
    )

    model = BertForSequenceClassification.from_pretrained(
        TEXT_MODEL_PATH
    )

    model.to(device)
    model.eval()

    return tokenizer, model

text_tokenizer, text_model = load_text_model()

# =========================================================
# IMAGE TRANSFORM
# =========================================================
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.5, 0.5, 0.5],
        [0.5, 0.5, 0.5]
    )
])

idx_to_class = {
    0: "Real",
    1: "Fake"
}

# =========================================================
# AUDIO FEATURE EXTRACTION
# =========================================================
def extract_audio(file_path):

    audio, sr = librosa.load(
        file_path,
        sr=16000
    )

    audio = librosa.util.fix_length(
        audio,
        size=32000
    )

    mel = librosa.feature.melspectrogram(
        y=audio,
        sr=sr,
        n_mels=128
    )

    mel = librosa.power_to_db(mel)

    mel = (
        mel - np.min(mel)
    ) / (
        np.max(mel) - np.min(mel) + 1e-6
    )

    mel = tf.image.resize(
        mel[..., np.newaxis],
        (128, 128)
    )

    return np.expand_dims(
        mel.numpy(),
        axis=0
    )

# =========================================================
# IMAGE PREDICTION
# =========================================================
def predict_image(image):

    img = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():

        outputs = image_model(img)

        probs = torch.softmax(outputs, dim=1)

    pred = torch.argmax(probs, dim=1).item()

    label = idx_to_class[pred]

    conf = probs[0][pred].item()

    return label, conf, img, pred

# =========================================================
# TEXT PREDICTION
# =========================================================
def predict_text(text):

    encoding = text_tokenizer(
        text,
        truncation=True,
        padding=True,
        max_length=256,
        return_tensors="pt"
    )

    input_ids = encoding["input_ids"].to(device)

    attention_mask = encoding["attention_mask"].to(device)

    with torch.no_grad():

        outputs = text_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        probs = torch.softmax(
            outputs.logits,
            dim=1
        )

    pred = torch.argmax(probs, dim=1).item()

    confidence = probs[0][pred].item()

    if pred == 0:
        label = "Human"
    else:
        label = "AI"

    return label, confidence

# =========================================================
# TEXT SHAP EXPLANATION
# =========================================================
def shap_text_explanation(text):

    def predictor(x):

        encodings = text_tokenizer(
            list(x),
            truncation=True,
            padding=True,
            max_length=256,
            return_tensors="pt"
        )

        with torch.no_grad():

            outputs = text_model(
                input_ids=encodings["input_ids"].to(device),
                attention_mask=encodings["attention_mask"].to(device)
            )

            probs = torch.softmax(
                outputs.logits,
                dim=1
            )

        return probs.cpu().numpy()

    explainer = shap.Explainer(
        predictor,
        shap.maskers.Text(text_tokenizer)
    )

    shap_values = explainer([text])

    return shap_values

# =========================================================
# IMAGE GRADCAM
# =========================================================
def gradcam(img_tensor, class_idx):

    grads = []
    acts = []

    def f_hook(m, i, o):
        acts.append(o)

    def b_hook(m, gi, go):
        grads.append(go[0])

    layer = image_model.layer4[-1].conv3

    fh = layer.register_forward_hook(f_hook)

    bh = layer.register_full_backward_hook(b_hook)

    out = image_model(img_tensor)

    image_model.zero_grad()

    out[0][class_idx].backward()

    g = grads[0].detach()

    a = acts[0].detach()

    w = torch.mean(g, dim=[0, 2, 3])

    for i in range(a.shape[1]):
        a[:, i, :, :] *= w[i]

    heatmap = torch.mean(a, dim=1).squeeze()

    heatmap = np.maximum(
        heatmap.cpu().numpy(),
        0
    )

    if np.max(heatmap) != 0:
        heatmap /= np.max(heatmap)

    fh.remove()
    bh.remove()

    return heatmap

# =========================================================
# AUDIO SALIENCY
# =========================================================
def audio_saliency(model, input_data):

    input_tensor = tf.convert_to_tensor(
        input_data
    )

    with tf.GradientTape() as tape:

        tape.watch(input_tensor)

        pred = model(input_tensor)

    grad = tape.gradient(
        pred,
        input_tensor
    )

    saliency = tf.abs(grad)[0]

    return saliency.numpy().squeeze()

# =========================================================
# IMAGE REASONING
# =========================================================
def image_reason(label):

    if label == "Real":

        return """
🟢 WHY REAL IMAGE?

✔ Natural facial texture consistency
✔ Smooth lighting and shadows
✔ No blending artifacts
✔ Realistic skin structure detected
"""

    return """
🔴 WHY FAKE IMAGE?

❌ Face blending artifacts detected
❌ Irregular texture patterns
❌ Inconsistent lighting
❌ GAN-style manipulation signals
"""

# =========================================================
# AUDIO REASONING
# =========================================================
def audio_reason(label):

    if label == "Real":

        return """
🟢 WHY REAL AUDIO?

✔ Stable waveform patterns
✔ Natural speech rhythm
✔ Clean frequency distribution
✔ No synthetic distortions detected
"""

    elif label == "Fake":

        return """
🔴 WHY FAKE AUDIO?

❌ Synthetic frequency artifacts
❌ Irregular spectrogram patterns
❌ AI-generated speech inconsistencies
❌ Distorted energy distribution
"""

    return """
⚠ Mixed signal patterns detected.
"""

# =========================================================
# TEXT REASONING
# =========================================================
def text_reason(label):

    if label == "Human":

        return """
🟢 WHY HUMAN TEXT?

✔ Natural writing variations
✔ Human-like sentence flow
✔ Diverse vocabulary usage
✔ Organic language structure
"""

    return """
🔴 WHY AI GENERATED TEXT?

❌ Repetitive sentence patterns
❌ Predictable word structure
❌ Machine-generated consistency
❌ AI-style language signals detected
"""

# =========================================================
# SIDEBAR
# =========================================================
st.sidebar.title("⚙ Detection Modes")

mode = st.sidebar.radio(
    "Choose Input Type",
    [
        "🎧 Audio",
        "🖼️ Image",
        "📝 Text"
    ]
)

# =========================================================
# AUDIO MODE
# =========================================================
if mode == "🎧 Audio":

    st.header("🎧 Audio Deepfake Detection")

    file = st.file_uploader(
        "Upload Audio File",
        type=["wav", "mp3"]
    )

    if file is not None:

        with open("temp.wav", "wb") as f:
            f.write(file.read())

        st.audio("temp.wav")

        features = extract_audio("temp.wav")

        pred = audio_model.predict(features)[0][0]

        st.subheader("📊 Confidence Score")

        st.progress(float(pred))

        st.write(f"Prediction Score: {pred:.4f}")

        if pred < 0.45:

            label = "Real"

            st.success("🟢 REAL AUDIO")

        elif pred > 0.60:

            label = "Fake"

            st.error("🔴 FAKE AUDIO")

        else:

            label = "Uncertain"

            st.warning("⚠ UNCERTAIN AUDIO")

        # AUDIO EXPLANATION
        if st.button("🔍 Explain Audio"):

            saliency = audio_saliency(
                audio_model,
                features
            )

            st.subheader("🎧 Audio Heatmap")

            fig, ax = plt.subplots(figsize=(10, 4))

            im = ax.imshow(
                saliency,
                cmap="coolwarm",
                aspect="auto"
            )

            ax.set_title(
                "Important Audio Regions"
            )

            plt.colorbar(im)

            st.pyplot(fig)

        st.subheader("🧠 Why this prediction?")

        st.info(audio_reason(label))

# =========================================================
# IMAGE MODE
# =========================================================
elif mode == "🖼️ Image":

    st.header("🖼️ Image Deepfake Detection")

    file = st.file_uploader(
        "Upload Image",
        type=["jpg", "jpeg", "png"]
    )

    if file is not None:

        image = Image.open(file).convert("RGB")

        col1, col2 = st.columns(2)

        with col1:
            st.image(
                image,
                caption="Uploaded Image",
                use_container_width=True
            )

        label, conf, img_tensor, pred_class = predict_image(image)

        with col2:

            st.subheader("📊 Confidence")

            st.progress(float(conf))

            st.write(
                f"Confidence: {conf*100:.2f}%"
            )

            if label == "Fake":

                st.error(
                    f"🔴 {label}"
                )

            else:

                st.success(
                    f"🟢 {label}"
                )

        if st.button("🔥 Generate Grad-CAM"):

            heatmap = gradcam(
                img_tensor,
                pred_class
            )

            img = np.array(
                image.resize((224, 224))
            )

            heatmap = cv2.resize(
                heatmap,
                (224, 224)
            )

            heatmap = np.uint8(
                255 * heatmap
            )

            heatmap = cv2.applyColorMap(
                heatmap,
                cv2.COLORMAP_JET
            )

            overlay = cv2.addWeighted(
                img,
                0.6,
                heatmap,
                0.4,
                0
            )

            st.image(
                overlay,
                caption="Grad-CAM Visualization",
                use_container_width=True
            )

        st.subheader("🧠 Why this prediction?")

        st.info(image_reason(label))

# =========================================================
# TEXT MODE
# =========================================================
elif mode == "📝 Text":

    st.header("📝 AI Text Detection")

    text_input = st.text_area(
        "Enter Text",
        height=250
    )

    if st.button("🔍 Analyze Text"):

        if len(text_input.strip()) == 0:

            st.warning(
                "Please enter some text."
            )

        else:

            label, confidence = predict_text(
                text_input
            )

            st.subheader("📊 Confidence")

            st.progress(float(confidence))

            st.write(
                f"Confidence: {confidence*100:.2f}%"
            )

            if label == "AI":

                st.error(
                    f"🔴 AI GENERATED TEXT"
                )

            else:

                st.success(
                    f"🟢 HUMAN WRITTEN TEXT"
                )

            st.subheader("🧠 Why this prediction?")

            st.info(text_reason(label))

            # =====================================
            # SHAP VISUALIZATION
            # =====================================
            st.subheader("🧠 SHAP Word Importance")

            with st.spinner(
                "Generating SHAP explanations..."
            ):

                shap_values = shap_text_explanation(
                    text_input
                )

                shap_html = shap.plots.text(
                    shap_values[:, :, 1],
                    display=False
                )

                st.components.v1.html(
                    shap_html,
                    height=400,
                    scrolling=True
                )

Overwriting app.py


#Running using ngrok

In [6]:
from pyngrok import ngrok

ngrok.set_auth_token("30u1DllnDwjArYZqt4747XIc7Ot_7JCfW4ULkKuJxnFZ2AuJN")

In [7]:
!streamlit run app.py &>/content/logs.txt &

In [8]:
public_url = ngrok.connect(8501)
print("🚀 Your Streamlit App URL:", public_url)

🚀 Your Streamlit App URL: NgrokTunnel: "https://4551-35-231-72-206.ngrok-free.app" -> "http://localhost:8501"
